# Coppia A di Training: ResNet18 + EfficientNetV2-S (ricetta `light`)

**Questo notebook allena SOLO due backbone** (ResNet18 ed EfficientNetV2-S) sotto la ricetta
`light`, e salva i pesi in `models/resnet18_light.pth` e `models/effv2s_light.pth`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

BASE = Path('/content/drive/MyDrive/2026_MLinf_gr41/Waste-Project')
DRIVE_DATASET_DIR = BASE / 'dataset'
DATASET_DIR       = Path('/content/dataset_local')
SPLIT_CSV         = BASE / 'splits' / 'split.csv'
MODELS_DIR        = BASE / 'models'
RESULTS_DIR       = BASE / 'results'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE    = 224
NUM_CLASSES = 8
SEED        = 1234
EPOCHS      = 15
RECIPE_TAG  = 'light'
SKIP_IF_EXISTS = True

CLASS_NAMES    = ['battery','clothing','glass','metal','organic','papery','plastic','undifferentiated']
TARGET_CLASSES = ['metal', 'plastic']
IMAGENET_MEAN  = [0.485, 0.456, 0.406]
IMAGENET_STD   = [0.229, 0.224, 0.225]

# Tutti i backbone del progetto
# Il sottoinsieme da allenare in questo notebook è definito da TRAIN_BACKBONES qui sotto
ALL_BACKBONES = {
    'resnet18'     : {'opt': 'sgd',   'micro': 64, 'accum': 1},
    'regnety16gf'  : {'opt': 'sgd',   'micro': 32, 'accum': 2},
    'effv2s'       : {'opt': 'sgd',   'micro': 16, 'accum': 4},
    'convnext_tiny': {'opt': 'adamw', 'micro': 32, 'accum': 2},
}

TRAIN_BACKBONES = {k: ALL_BACKBONES[k] for k in ['resnet18', 'effv2s']}

FAMILIES    = ['geometric', 'acquisition', 'background', 'resolution']
INTENSITIES = ['mild', 'moderate']

Mounted at /content/drive


In [ ]:
import io, os, random, time, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image, ImageEnhance, ImageFilter
from tqdm.auto import tqdm
from sklearn.metrics import balanced_accuracy_score, recall_score
import matplotlib.pyplot as plt

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, '|', torch.cuda.get_device_name(0) if device.type=='cuda' else 'CPU')

if not DATASET_DIR.exists():
    print('Copio il dataset in locale...')
    t0 = time.time(); shutil.copytree(DRIVE_DATASET_DIR, DATASET_DIR)
    print(f'Fatto in {time.time()-t0:.0f}s')
else:
    print('Copia locale gia presente.')

Device: cuda | Tesla T4
Copio il dataset in locale...
Fatto in 398s


In [ ]:
df_all = pd.read_csv(SPLIT_CSV)
df_train = df_all[df_all['split'] == 'train'].reset_index(drop=True).copy()
df_val   = df_all[df_all['split'] == 'val'].reset_index(drop=True).copy()
print(f"Train: {len(df_train)}  |  Val: {len(df_val)}")

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
preprocess = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

Train: 12413  |  Val: 3102


In [ ]:
def _rng(idx, family, intensity):
    fam_id = {'geometric':1, 'acquisition':2, 'background':3, 'resolution':4}[family]
    int_id = {'mild':1, 'moderate':2}[intensity]
    return np.random.RandomState((SEED * 100003 + idx * 97 + fam_id * 13 + int_id) % (2**32))

def perturb_geometric(img, idx, intensity):
    r = _rng(idx, 'geometric', intensity)
    max_rot, min_scale = (8.0, 0.85) if intensity == 'mild' else (15.0, 0.60)
    W, H = img.size
    ang = r.uniform(-max_rot, max_rot)
    img = img.rotate(ang, resample=Image.BILINEAR, expand=False, fillcolor=(255, 255, 255))
    scale = r.uniform(min_scale, 1.0)
    cw, ch = max(1, int(W * np.sqrt(scale))), max(1, int(H * np.sqrt(scale)))
    x0 = r.randint(0, max(1, W - cw + 1)); y0 = r.randint(0, max(1, H - ch + 1))
    return img.crop((x0, y0, x0 + cw, y0 + ch)).resize((W, H), Image.BILINEAR)

def perturb_acquisition(img, idx, intensity):
    r = _rng(idx, 'acquisition', intensity)
    blur, q, jit, noise = (0.6, 70, 0.10, 4.0) if intensity == 'mild' else (1.2, 40, 0.20, 10.0)
    img = img.filter(ImageFilter.GaussianBlur(radius=blur * r.uniform(0.7, 1.3)))
    for Enh in (ImageEnhance.Brightness, ImageEnhance.Contrast, ImageEnhance.Color):
        img = Enh(img).enhance(1.0 + r.uniform(-jit, jit))
    buf = io.BytesIO(); img.save(buf, format='JPEG', quality=int(q)); buf.seek(0)
    img = Image.open(buf).convert('RGB')
    arr = np.asarray(img).astype(np.float32) + r.normal(0, noise, (img.size[1], img.size[0], 3))
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

def perturb_resolution(img, idx, intensity):
    r = _rng(idx, 'resolution', intensity)
    f = (0.5 if intensity == 'mild' else 0.34) * r.uniform(0.9, 1.1)
    W, H = img.size
    small = img.resize((max(1, int(W * f)), max(1, int(H * f))), Image.BILINEAR)
    return small.resize((W, H), Image.BILINEAR)

def perturb_background(img, idx, intensity):
    keep = 0.80 if intensity == 'mild' else 0.65
    W, H = img.size
    cw, ch = int(W * keep), int(H * keep)
    x0, y0 = (W - cw) // 2, (H - ch) // 2
    canvas = Image.new('RGB', (W, H), (255, 255, 255))
    canvas.paste(img.crop((x0, y0, x0 + cw, y0 + ch)), (x0, y0))
    return canvas

PERTURB = {'geometric': perturb_geometric, 'acquisition': perturb_acquisition,
           'background': perturb_background, 'resolution': perturb_resolution}

In [ ]:
def build_backbone(name):
    if name == 'resnet18':
        m = models.resnet18(weights='DEFAULT');           m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    elif name == 'regnety16gf':
        m = models.regnet_y_1_6gf(weights='DEFAULT');      m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    elif name == 'effv2s':
        m = models.efficientnet_v2_s(weights='DEFAULT');   m.classifier[1] = nn.Linear(m.classifier[1].in_features, NUM_CLASSES)
    elif name == 'convnext_tiny':
        m = models.convnext_tiny(weights='DEFAULT');       m.classifier[2] = nn.Linear(m.classifier[2].in_features, NUM_CLASSES)
    else:
        raise ValueError(name)
    return m

class TrainDataset(Dataset):
    def __init__(self, df): self.items = list(zip(df['filepath'].tolist(), df['label'].tolist()))
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        fp, lab = self.items[i]
        return train_tf(Image.open(DATASET_DIR / fp).convert('RGB')), lab

class ValDataset(Dataset):
    def __init__(self, df, family=None, intensity=None):
        self.items = list(zip(df['filepath'].tolist(), df['label'].tolist()))
        self.family, self.intensity = family, intensity
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        fp, lab = self.items[i]
        img = Image.open(DATASET_DIR / fp).convert('RGB')
        if self.family is not None:
            img = PERTURB[self.family](img, i, self.intensity)
        return preprocess(img), lab

@torch.no_grad()
def evaluate(model, df, family=None, intensity=None, bs=64):
    model.eval()
    dl = DataLoader(ValDataset(df, family, intensity), batch_size=bs, shuffle=False, num_workers=2)
    ys, ps = [], []
    for x, y in dl:
        ps.append(model(x.to(device)).argmax(1).cpu().numpy()); ys.append(np.asarray(y))
    y = np.concatenate(ys); p = np.concatenate(ps)
    bal = balanced_accuracy_score(y, p)
    tpr = recall_score(y, p, labels=list(range(NUM_CLASSES)), average=None, zero_division=0)
    return bal, dict(zip(CLASS_NAMES, tpr))

In [ ]:
def train_one(name, cfg):
    set_seed(SEED)
    model = build_backbone(name).to(device)
    if cfg['opt'] == 'sgd':
        opt = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)
    else:
        opt = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.05)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    crit  = nn.CrossEntropyLoss()
    micro, accum = cfg['micro'], cfg['accum']

    train_dl = DataLoader(TrainDataset(df_train), batch_size=micro, shuffle=True,
                          num_workers=2, drop_last=True)
    best_bal, best_state, peak_mem = -1.0, None, 0.0

    for epoch in range(EPOCHS):
        model.train()
        if epoch == 0 and device.type == 'cuda':
            torch.cuda.reset_peak_memory_stats()
        opt.zero_grad()
        for i, (x, y) in enumerate(tqdm(train_dl, desc=f"{name} ep{epoch+1}/{EPOCHS}", leave=False)):
            x, y = x.to(device), y.to(device)
            loss = crit(model(x), y) / accum
            loss.backward()
            if (i + 1) % accum == 0:
                opt.step(); opt.zero_grad()
        if len(train_dl) % accum != 0:
            opt.step(); opt.zero_grad()
        sched.step()
        if epoch == 0 and device.type == 'cuda':
            peak_mem = torch.cuda.max_memory_allocated() / 1e9
        bal, _ = evaluate(model, df_val)
        if bal > best_bal:
            best_bal = bal
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f"  {name} epoch {epoch+1:2d}: val balAcc = {bal:.4f}  (best {best_bal:.4f})")

    path = MODELS_DIR / f"{name}_{RECIPE_TAG}.pth"
    torch.save(best_state, path)
    print(f"  -> salvato {path.name} | best val balAcc {best_bal:.4f} | train peak {peak_mem:.2f} GB")
    del model; torch.cuda.empty_cache()
    return best_bal, peak_mem

In [ ]:
train_summary = {}
for name in TRAIN_BACKBONES:
    cfg  = ALL_BACKBONES[name]
    path = MODELS_DIR / f"{name}_{RECIPE_TAG}.pth"
    if SKIP_IF_EXISTS and path.exists():
        print(f"[skip] {path.name} gia presente."); continue
    print(f"\n===== Training {name} sotto '{RECIPE_TAG}' =====")
    bal, mem = train_one(name, cfg)
    train_summary[name] = {'best_val_balacc': round(bal, 4), 'train_peak_gb': round(mem, 2)}